# ATLAS (Adaptive Tiling of Localized Anisotropic Subspaces) — 9-PDE Benchmark Suite
### Dual-GPU Parallel Accelerated Execution on Kaggle (NVIDIA Tesla T4 x2)

This notebook clones the repository directly from GitHub, detects available GPUs, and runs the complete **ATLAS** multi-model benchmark suite in parallel across **2x T4 GPUs** (cutting total execution time in half):
- **Worker 0 (GPU 0 / cuda:0):** Burgers 1D, 2D Convection, 2D Helmholtz, 2D Cavity Navier-Stokes, 2D Kovasznay Flow (~8,000 epochs)
- **Worker 1 (GPU 1 / cuda:1):** 2D Quantum NLS, 4D Diffusion, 2D Bratu Ignition, 2D Klein-Gordon Wave (~7,000 epochs)

Saves all PyTorch checkpoints (`.pt`), raw loss trajectories (`.json`), and renders Rougier-standard publication figures automatically.

**Hardware Recommendation:** In Kaggle sidebar, set **Settings → Accelerator → GPU T4 x2** and **Internet → On**.

In [ ]:
# Step 0: Clone the Repository directly from GitHub (Zero Uploads Required)
import os

if os.path.exists("benchmarks/run_master_benchmarks.py"):
    print("[*] Already inside repository directory. Pulling latest commits...")
    !git pull origin main
else:
    if not os.path.exists("atlas-neural-amr"):
        print("[*] Cloning repository from GitHub...")
        !git clone https://github.com/KartikeyaGangwar/atlas-neural-amr.git
    %cd atlas-neural-amr
    !git pull origin main

print("[+] Working directory set:", os.getcwd())

In [ ]:
# Step 1: Verify Multi-GPU Hardware Acceleration
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    print(f'Total Detected GPUs: {n_gpu}')
    for i in range(n_gpu):
        print(f'  [GPU {i}] {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)')
else:
    print('WARNING: Please enable GPU accelerator in Kaggle settings (Settings -> Accelerator -> GPU T4 x2)')

In [ ]:
# Step 2: Install required packages
!pip install --quiet matplotlib scipy numpy

In [ ]:
# Step 3: Run Full Benchmark Suite in Parallel across both GPUs
# (Automatically divides PDEs across GPU 0 and GPU 1 simultaneously)
!python -u benchmarks/run_parallel_benchmarks.py

In [ ]:
# Step 4: Display Generated Rougier-Standard Figures Inline
import glob
from IPython.display import Image, display

plots = sorted(glob.glob('results/plots/*.png'))
print(f'Total Generated High-Resolution Figures: {len(plots)}')
for p in plots:
    print(f'\n--- Figure: {p} ---')
    display(Image(filename=p, width=850))

In [ ]:
# Step 5: Package All Checkpoints, Results, and Figures into a Downloadable Zip
import os
from IPython.display import FileLink, display

zip_path = "/kaggle/working/atlas_checkpoints_and_results.zip" if os.path.exists("/kaggle/working") else "atlas_checkpoints_and_results.zip"
!zip -r {zip_path} checkpoints/ results/

print(f"\n[+] Download package ready: {zip_path}")
print("[+] Click the link below to download all checkpoints and publication figures directly:")
display(FileLink(zip_path))